[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/07_unsloth.ipynb)

# Unsloth (accelerator)

> **Fine-tuning series — 07 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## 4. Unsloth  ·  *accelerator*  (CUDA only)

Not a new method — it runs LoRA/QLoRA **~2x faster, ~½ the VRAM** via fused kernels.
Here it's QLoRA-instruction-tuning, same job as §3, the Unsloth way.

In [ ]:
from unsloth import FastLanguageModel

model, uns_tok = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=1024, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

uns_ds = Dataset.from_list([{"text": uns_tok.apply_chat_template(
    [{"role": "user", "content": e["instruction"]},
     {"role": "assistant", "content": e["output"]}], tokenize=False)}
    for e in instructions])

trainer = SFTTrainer(
    model=model, tokenizer=uns_tok, train_dataset=uns_ds,
    args=SFTConfig(output_dir="ft_unsloth", dataset_text_field="text",
                   max_length=1024, per_device_train_batch_size=2,
                   max_steps=20, learning_rate=2e-4, logging_steps=5,
                   optim="adamw_8bit", report_to="none"))
trainer.train()
del trainer, model; cleanup()